# Проект: Дашборд конверсий

## Подготовка к работе с данными
Вы работаете аналитиком в образовательной платформе. Разработчики для вас сделали выгрузку данных о посещениях и регистрациях.

## Посещения

**uuid - id** посетителя

**platform** - платформа с которой был заход на сайт: web, ios, android, bot

**user_agen**t - User-Agent посещения

**date** - дата посещения сайта

## Регистрации

**date** - дата регистрации на сайте

**user_id** - id зарегистрированного пользователя

**email** - почта пользователя

**platform** - платформа с которой была сделана регистрация: web, ios, android

**registration_type** - тип регистрации: email, google, yandex, apple

## Задачи

Склонируйте созданный репозиторий проекта на рабочую машину
Создайте Jypyter Notebook и подключите необходимые библиотеки (pandas, requests)
Скачайте и подгрузите данные по указанным ссылкам
Изучите данные, сделайте предварительный анализ с помощью dataframe.describe

## В этом проекте вы можете использовать только следующие библиотеки:

- pandas
- matplotlib
- numpy
- seaborn
- requests
- plotly
- python-dotenv

## Ссылки
Ниже даны выгрузки данных

[Визиты](https://drive.google.com/file/d/1QosQQ4RRNR9rkL4t7sB707h2Uy0XfYJe/view?usp=drive_link) - тысяча записей с визитами

[Регистрации](https://drive.google.com/file/d/1AeQz0kaSgz0lxYSDtuNm36muhy5fRCzZ/view?usp=drive_link) - тысяча записей о первых регистрациях

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import plotly.graph_objects as go
import plotly.express as px
from dotenv import load_dotenv

## Запросы к API

Ранее мы работали со статичными данными в файлах. Теперь поработаем с данными, которые получаем в реальном времени по сети.

### Описание API

У API есть два маршрута со следующими параметрами:

- Запрос на данные по посещениям от даты `START` до даты `END`:
    
    ```
    /visits?begin=START&end=END
    ```
    
- Запрос на данные по регистрациям c даты `START` по дату `END`:
    
    ```
    /registrations?begin=START&end=END
    ```
    

Формат даты должен быть вида `YYYY-MM-DD`, например `2023-04-23`

---

Запрос по пути `/visits?begin=START&end=END` вернет данные в формате JSON с массивом записей следующего вида:

- `visit_id` - id посетителя
- `platform` - платформа с которой был заход на сайт: `web`, `ios`, `android`, `bot`
- `user_agent` - User-Agent посещения
- `datetime` - дата посещения сайта

К примеру:

```
GET https://data-charts-api.hexlet.app/visits?begin=2022-06-01&end=2023-06-01

[
    {
        "datetime": "Wed, 01 Mar 2023 13:29:22 GMT",
        "platform": "web",
        "user_agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2) AppleWebKit/601.3.9 (KHTML, like Gecko) Version/9.0.2 Safari/601.3.9",
        "visit_id": "1de9ea66-70d3-4a1f-8735-df5ef7697fb9"
    },
    {
        "datetime": "Wed, 01 Mar 2023 16:44:28 GMT",
        "platform": "web",
        "user_agent": "Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/51.0.2704.64 Safari/537.36",
        "visit_id": "f149f542-e935-4870-9734-6b4501eaf614"
    }
]
```

Запрос по пути `/registrations?begin=START&end=END` вернет данные в формате JSON с массивом записей следующего вида:

- `user_id` - id зарегистрированного пользователя
- `email` - почта пользователя
- `platform` - платформа с которой была регистрация: `web`, `ios`, `android`
- `registration_type` - тип регистрации: `google`, `apple`, `email`, `yandex`
- `datetime` - дата регистрации на сайте

Например:

```
GET https://data-charts-api.hexlet.app/registrations?begin=2022-06-01&end=2023-06-02

[
    {
        "datetime": "Wed, 01 Mar 2023 00:25:39 GMT",
        "email": "joseph95@example.org",
        "registration_type": "google",
        "platform": "web",
        "user_id": "8838849"
    },
    {
        "datetime": "Wed, 01 Mar 2023 14:53:01 GMT",
        "email": "janetsuarez@example.net",
        "registration_type": "yandex",
        "platform": "web",
        "user_id": "8741065"
    }
]
```

### Ссылки

- [requests](https://requests.readthedocs.io/en/latest/) - библиотека для запросов по сети

### Задачи

- Запросите данные по API за период `2023-03-01 -> 2023-09-01`
- Сохраните ваш ноутбук в репозитории

In [ ]:
url_visits = "https://data-charts-api.hexlet.app/visits"
url_registrations = "https://data-charts-api.hexlet.app/registrations"

params = {
    "begin": "2023-03-01",
    "end": "2023-09-01"
}

In [ ]:
response_visits = requests.get(url_visits, params=params)
response_registrations = requests.get(url_registrations, params=params)

data_visits = response_visits.json()
data_registrations = response_registrations.json()
#print(data_visits[:2])

df_visits = pd.DataFrame(data_visits)
#print(df_visits.head())

df_registrations = pd.DataFrame(data_registrations)
#print(df_registrations.head())

## Расчет метрик

Для этого и последующих шагов используем данные, полученные из API на предыдущем шаге.

В этом шаге расчитаем конверсию визитов в регистрации. На выходе у вас должен получиться датафрейм со следующими полями:

- date_group — дата
- platform — платформа: `web`, `ios`, `android`
- visits — визиты в дату
- registrations — регистрации в дату
- conversion — конверсия

Среди визитов есть боты — это поисковые и SEO-боты, которые регулярно сканируют страницы. Определить их можно по слову `bot` в User-Agent.

Сагрегируем данные по дате и платформам. Также сохраним полученный датафрейм в формате JSON.

## Требования

- Данные должны быть отсортированы по дате, от ранних к более поздним
- Визиты ботов не должны влиять на расчет конверсии.

## Задачи

Выполните Jupyter Notebook следующие задачи:

- Сгруппируйте данные визитов по датам и платформам
- Сгруппируйте также данные регистраций по датам и платформам
- Объедините датайфреймы, сделайте итоговый датафрейм с расчетом конверсии
- Сохраните датафрейм в формате JSON _conversion.json_
- Поля датафрейма:
    - date_group - сагрегированный столбец дат
    - platform - платформа (`android`,`web`,`ios`)
    - visits - визиты за период `date_group`
    - registrations - регистрации за период `date_group`
    - conversion - конверсия по платформе

## Подсказки

- Пользователи могли заходить на сайт несколько раз, прежде чем зарегистрироваться. Учтите это и возьмите только последний визит для каждого `visit_id`
- Для сохранения датафрейма в JSON используйте метод `.to_json(./conversion.json)`
- Пример JSONа:
{
  "date_group": {
    "0": 1677628800000,
    "1": 1677628800000,
  },
  "platform": {
    "0": "android",
    "1": "ios",
  },
  "visits": {
    "0": 158,
    "1": 70,
  },
  "registrations": {
    "0": 120,
    "1": 59,
  },
  "conversion": {
    "0": 75.9493670886,
    "1": 84.2857142857,
  }
}

In [ ]:
# Обработка визитов: последний визит на visit_id
df_visits['datetime'] = pd.to_datetime(df_visits['datetime'])
df_visits_unique = (
    df_visits
    .sort_values('datetime')
    .drop_duplicates(subset='visit_id', keep='last')
    .copy()
)

# ИСПРАВЛЕНИЕ: группируем по дням, обнуляя время, но сохраняя тип datetime
df_visits_unique['date_group'] = df_visits_unique['datetime'].dt.normalize()

visits_agg = (
    df_visits_unique
    .groupby(['date_group', 'platform'], observed=True)
    .size()
    .reset_index(name='visits')
)

# Агрегация регистраций по дням и платформе
df_reg['datetime'] = pd.to_datetime(df_reg['datetime'])
# ИСПРАВЛЕНИЕ: тоже нормализуем, чтобы дни совпадали
df_reg['date_group'] = df_reg['datetime'].dt.normalize()

reg_agg = (
    df_reg
    .groupby(['date_group', 'platform'], observed=True)
    .size()
    .reset_index(name='registrations')
)

# Объединение и конверсия
df_merged = visits_agg.merge(reg_agg, on=['date_group', 'platform'], how='left')
df_merged['registrations'] = df_merged['registrations'].fillna(0).astype(int)

# Векторизованный расчёт конверсии (быстрее и чище, чем apply)
df_merged['conversion'] = df_merged['registrations'] / df_merged['visits']
df_merged['conversion'] = (df_merged['conversion'] * 100).fillna(0)

# Сохранение в JSON (timestamp-даты, orient=columns)
df_merged.to_json('./conversion.json', orient='columns', force_ascii=False)

print(df_merged[['date_group', 'platform', 'visits', 'registrations', 'conversion']].head(15))

## Добавляем рекламы

В этом шаге добавим данные по рекламным кампаниям. На выходе получим датафрейм со следующими полями:

- date_group — дата
- visits — визиты в дату
- registrations — регистрации в дату
- cost — затраты на рекламу, 0 если не было затрат
- utm_campaign — название рекламной кампании, `none` если не было в этот период рекламы

Сагрегируем данные по дате и сохраним их в JSON

### Описание CSV

CSV-таблица содержит записи следующего вида:

- `date` — дата проведения кампании
- `utm_source` — utm-источник: `yandex`, `vk`, `google`, `youtube`, `tg`
- `utm_medium` — utm-медиум: `cpc` или `social`
- `utm_campaign` — название кампании
- `cost` — затраты на рекламу

Например:

```
2023-03-01T09:16:57,google,cpc,virtual_reality_workshop,238
2023-03-02T12:48:25,google,cpc,virtual_reality_workshop,164
2023-04-25T11:30:16,google,cpc,full_stack_dev_challenge,266
```

## Требования

- Данные должны быть отсортированы по дате, от ранних к более поздним

## Ссылки

- [ads](https://drive.google.com/file/d/12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj/view?usp=sharing) - данные о рекламах

## Задачи

Выполните в Jupyter Notebook следующие задачи:

- Объедините датайфрейм конверсий с рекламными кампаниями
- Сохраните датафрейм в формате JSON с именем _ads.json_

In [ ]:
# --- Подготовка рекламы (агрегация по дате) ---
df_ads['date'] = pd.to_datetime(df_ads['date'])
#df_ads['date_group'] = df_ads['date'].dt.date
# Делаем date_group как datetime (полночь дня) вместо date
df_ads['date_group'] = df_ads['date'].dt.normalize()  # это даст datetime64[ns], время = 00:00:00


ads_agg = (
    df_ads
    .groupby('date_group', observed=True)
    .agg(
        cost=('cost', 'sum'),
        utm_campaign=('utm_campaign', lambda x: x.iloc[0] if len(x) > 0 else 'none')
    )
    .reset_index()
)

ads_agg = ads_agg.sort_values('date_group')

# ---  Объединение с конверсиями ---
df_final = df_merged.merge(ads_agg, on='date_group', how='left')

# Заполняем отсутствующие значения (если в день не было рекламы)
df_final['cost'] = df_final['cost'].fillna(0)
df_final['utm_campaign'] = df_final['utm_campaign'].fillna('none')

# Сортировка: сначала по дате, потом по платформе
df_final = df_final.sort_values(['date_group', 'platform'])
print(df_final.head())

# --- Сохранение в JSON ---
df_final.to_json('./ads.json', orient='records', force_ascii=False)


## Визуализация

В этом шаге визуализируем наши расчеты. Построим следующие графики в формате PNG:

- Итоговые визиты
- Итоговые визиты с разбивкой по платформам: `web`, `android`, `ios`
- Итоговые регистрации
- Итоговые регистрации с разбивкой по платформе: `web`, `android`, `ios`
- Конверсия по каждой платформе
- Средняя конверсия
- Стоимости реклам
- Визиты за весь период с цветовым выделением рекламной кампании
- Регистрации за весь период с цветовым выделением рекламной кампании

## Задачи

Выполните Jupyter Notebook следующие задачи:

- Установите библиотеки для визуализации
- Постройте требуемые графики
- Сохраните их в PNG в директорию _./charts_, например как `plt.savefig('./charts/registrations_by_platform.png')`

## Примеры графиков

- [Итоговые визиты](https://cdn6.hexlet.io/EGQ0jWhBQ385.png)
- [Итоговые визиты с разбивкой по платформам](https://cdn6.hexlet.io/JcqGjT4zNLbY.png)
- [Итоговые регистрации](https://cdn6.hexlet.io/0wjDFws6IpZm.png)
- [Итоговые регистрации с разбивкой по платформе](https://cdn6.hexlet.io/e8d4m1cKbXGw.png)
- [Итоговые конверсии](https://cdn6.hexlet.io/DmVBXqxdwjzV.png)
- [Конверсия по каждой платформе](https://cdn6.hexlet.io/OO7xfsJE6Y0C.png)
- [Стоимости реклам](https://cdn6.hexlet.io/8mqJ9glmdQyh.png)
- [Визиты и регистрации с выделением рекламных кампаний](https://cdn6.hexlet.io/1N0oDZm6DQr3.png)